# **Import**

In [1]:
# Install required packages
%pip install -q datasets
%pip install -q transformers
%pip install -q sentence-transformers


import os
import sys
from datasets import load_dataset
import json
from transformers import pipeline, set_seed
import random
import torch
from huggingface_hub import login
import getpass
from tqdm import tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


# **Seed to ensure reproducibility**

In [2]:
def set_seeds(seed=42):

    random.seed(seed)
    # In caso dovessimo usare Numpy
    # np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
        
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    # Seed on transformers library
    set_seed(seed)
    

set_seeds(seed=42)

# **Hugging Face Login**

In [3]:
# Insert the hugging face token
print("Please enter your Hugging Face Access Token: (You must have access to Llama 3.2):")
hf_token = getpass.getpass() 
login(token=hf_token)

Please enter your Hugging Face Access Token: (You must have access to Llama 3.2):


# **Environment Setup**

In [4]:
# Automatic Environment Detection
if 'google.colab' in sys.modules:
    print("🌐 Environment detected: Google Colab")
    
    # Automatically mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    BASE_DIR = '/content/drive/MyDrive/MNLP_2026_HW2_2278125_2283937/' 

else:
    print("💻 Environment detected: Local Machine")
    
    BASE_DIR = 'C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026' 

# Defining paths for subdirectories 
JSON_DIR = os.path.join(BASE_DIR, 'JSON')
PLOTS_DIR = os.path.join(BASE_DIR, 'plots')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')

# Creating directories if they don't exist 
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"📁 Base Directory set to: {BASE_DIR}")

# Set up hardware acceleration (GPU if available, else CPU)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected: {device}")

💻 Environment detected: Local Machine
📁 Base Directory set to: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026
Device selected: cuda


# **Dataset Loading and Exploration**

In [5]:
ds = load_dataset(
    "sapienzanlp-course-materials/hw-mnlp-2026"
)

# Sanity check
for split in ds.keys():
    print(f"Split: {split:6} | Features: {ds[split].column_names}")

# Unpack splits into dedicated variables for easier access
train = ds['train']
test = ds['test']
blind = ds['blind']

Split: test   | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']
Split: blind  | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']
Split: train  | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']


# **Rankings Loading**

In [6]:
# Rankings file name definition
rankings_test_filename = 'hit_k_1-test-distilbert-triplet-cosine_(reranking).jsonl'
rankings_blind_filename = 'hit_k_1-blind-distilbert-triplet-cosine_(reranking).jsonl'

# Complete path creation
rankings_test_path = os.path.join(JSON_DIR, rankings_test_filename)
rankings_blind_path = os.path.join(JSON_DIR, rankings_blind_filename)

retrieved_results_test = []
retrieved_results_blind = []

# Test set Rankings loading
try:
    with open(rankings_test_path, 'r', encoding='utf-8') as f:
        for line in f:
            retrieved_results_test.append(json.loads(line.strip()))
            
    print(f"✅ Test Set rankings loaded with success!")
    print(f"   📍 File: {rankings_test_path}")

except FileNotFoundError:
    print(f"❌ ERROR: TEST SET rankings does not found:\n{rankings_test_path}\n")


# Blind set Rankings loading
try:
    with open(rankings_blind_path, 'r', encoding='utf-8') as f:
        for line in f:
            retrieved_results_blind.append(json.loads(line.strip()))
            
    print(f"✅ Blind Set rankings loaded with success!")
    print(f"   📍 File: {rankings_blind_path}")

except FileNotFoundError:
    print(f"❌ ERROR: BLIND SET rankings does not found:\n{rankings_blind_path}")

# Unique Dictionary creation {'query_id1': [rankings], 'query_id2': [rankings], ...}
rankings_test = {}
for item in retrieved_results_test:
    rankings_test.update(item)

rankings_blind = {}
for item in retrieved_results_blind:
    rankings_blind.update(item)

✅ Test Set rankings loaded with success!
   📍 File: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\JSON\hit_k_1-test-distilbert-triplet-cosine_(reranking).jsonl
✅ Blind Set rankings loaded with success!
   📍 File: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\JSON\hit_k_1-blind-distilbert-triplet-cosine_(reranking).jsonl


# **Small LMs Implementation**

In [7]:

# Function to generate prompts in RAG and Oracle settings for both test and blind sets
def prompts_generation(dataset, rankings, k, oracle=False):
        
        all_prompts = []
        all_retrieved_chunks = []

        queries = dataset['query']
        query_ids = dataset['query_id']
        candidate_chunks = dataset['candidate_chunks']

        blind = all(pos == 100 for pos in dataset['answer_pos'])

        # Mapping query_ids to their corresponding answer positions (only for non-blind datasets)
        if not blind:
            mapped_answers = dict(zip(query_ids, dataset['answer_pos']))
        else:
            mapped_answers = {}
            if oracle:
                print("⚠️ Note: Blind Dataset detected (all values are 100). Oracle mode will be ignored to avoid errors.")

        # Generating prompts for each query
        for q_id, q_text in zip(query_ids, queries):

            # Retrieving the top-k chunk ids for the current query
            best_chunk_positions = rankings.get(q_id, [])[:k]
            
            context_block = ""
            
            # We iterate over the top-k chunk positions to build the context block for the prompt
            for i, c_position in enumerate(best_chunk_positions, 1):

                # If oracle is True, we replace the top-ranked chunk with the answer
                if i==1 and oracle and q_id in mapped_answers:
                    answer_pos = mapped_answers[q_id]
                    best_chunk_positions[0] = answer_pos
                    chunk_text = candidate_chunks[answer_pos]
                else:
                    chunk_text = candidate_chunks[c_position]
                
                context_block += f"{i}. {chunk_text}\n"
            
            # Writing the prompt with the retrieved context and the query
            full_prompt = (
            f"Given the following information:\n"
            f"{context_block}"
            f"Reply to this question:\n"
            f"{q_text}"
            )

            all_prompts.append(full_prompt)
            all_retrieved_chunks.append(best_chunk_positions)
            
        return all_prompts, all_retrieved_chunks

# Function to generate answers for a given dataset and generator
def answers(dataset, generator, settings='Baseline', k=3):
    
    answers = {}

    split = str(dataset.split).capitalize() 
    
    # We get all the queries to work with batches
    queries = list(dataset['query'])
    query_ids = list(dataset['query_id'])

    # Depending on the settings, we either use the raw queries (Baseline) or generate prompts with retrieved context (RAG and Oracle)
    if settings == 'Baseline':   
        prompts = queries
        retrieved_chunks = [[] for _ in range(len(queries))]
    elif settings in ['RAG', 'Oracle']:
        prompts, retrieved_chunks = prompts_generation(dataset, rankings_test if split=='Test' else rankings_blind, k=k, oracle=(settings=='Oracle'))
    else:
        raise ValueError(f"Invalid settings: {settings}. Choose from 'Baseline', 'RAG', or 'Oracle'.")
    
    
    # Batch generation of answers using the provided generator pipeline
    print(f"🔍 Starting generation for {len(queries)} queries in {split} set with {settings} setting...")
    outputs = generator(
        prompts,
        batch_size=16,          
        max_new_tokens=50,  
        truncation=True,    
        return_full_text=False,  
        do_sample=False,         
        pad_token_id=generator.tokenizer.eos_token_id 
    )
    print(f"✅ Generation completed for {len(queries)} queries in {split} set with {settings} setting.")
     
    # We iterate over the generated outputs and store them in the answers dictionary
    for q_id, out, prompt, chunks in zip(query_ids, outputs, prompts, retrieved_chunks):
        
        generated_answer = out[0]['generated_text'].strip()
        
        answers[q_id] = {
            "query_id": q_id,
            "retrieved_chunks": chunks,
            "augmented_prompt": prompt if settings in ["RAG", "Oracle"] else None,
            "generated_answer": generated_answer
        }

    return answers

## **Llama-3.2-3B-Instruct**

In [8]:
torch.cuda.empty_cache()

llama_id = "meta-llama/Llama-3.2-3B-Instruct"
llama = pipeline(
    "text-generation", 
    model=llama_id, 
    dtype=torch.bfloat16, 
    device=0, 
    token = True
)

llama.tokenizer.pad_token = llama.tokenizer.eos_token
llama.tokenizer.padding_side = 'left'
llama.tokenizer.model_max_length = 512
llama.generation_config.max_length = None

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

### **Baseline**

In [9]:
Llama_baseline_test_answers = answers(test, generator=llama)
print("✅ Llama Baseline Test Answers Generation Completed!")
Llama_baseline_blind_answers = answers(blind, generator=llama)
print("✅ Llama Baseline Blind Answers Generation Completed!")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔍 Starting generation for 2000 queries in Test set with Baseline setting...
✅ Generation completed for 2000 queries in Test set with Baseline setting.
✅ Llama Baseline Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with Baseline setting...
✅ Generation completed for 1322 queries in Blind set with Baseline setting.
✅ Llama Baseline Blind Answers Generation Completed!


### **RAG**

In [10]:
Llama_RAG_test_answers = answers(test, generator=llama, settings='RAG', k=3)
print("✅ Llama RAG Test Answers Generation Completed!")
Llama_RAG_blind_answers = answers(blind, generator=llama, settings='RAG', k=3)
print("✅ Llama RAG Blind Answers Generation Completed!")

🔍 Starting generation for 2000 queries in Test set with RAG setting...
✅ Generation completed for 2000 queries in Test set with RAG setting.
✅ Llama RAG Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with RAG setting...
✅ Generation completed for 1322 queries in Blind set with RAG setting.
✅ Llama RAG Blind Answers Generation Completed!


### **Oracle**

In [11]:
Llama_Oracle_test_answers = answers(test, generator=llama, settings='Oracle', k=3)
print("✅ Llama Oracle Test Answers Generation Completed!")
Llama_Oracle_blind_answers = answers(blind, generator=llama, settings='Oracle', k=3)
print("✅ Llama Oracle Blind Answers Generation Completed!")

🔍 Starting generation for 2000 queries in Test set with Oracle setting...
✅ Generation completed for 2000 queries in Test set with Oracle setting.
✅ Llama Oracle Test Answers Generation Completed!
⚠️ Note: Blind Dataset detected (all values are 100). Oracle mode will be ignored to avoid errors.
🔍 Starting generation for 1322 queries in Blind set with Oracle setting...
✅ Generation completed for 1322 queries in Blind set with Oracle setting.
✅ Llama Oracle Blind Answers Generation Completed!


## **Qwen3-0.6B**

In [ ]:
torch.cuda.empty_cache()

qwen_id = "Qwen/Qwen3-0.6B"
qwen = pipeline(
    "text-generation", 
    model=qwen_id, 
    dtype=torch.bfloat16, 
    device=0, 
    token = True
)

qwen.generation_config.max_length = None

[GPU prima del cache clear]
  Allocata:  5.99 GB
  Riservata: 8.55 GB
  Libera:    7.37 GB
  Totale:    15.92 GB
[GPU dopo il cache clear]
  Allocata:  5.99 GB
  Riservata: 6.05 GB
  Libera:    9.87 GB
  Totale:    15.92 GB


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

c:\Users\djmat\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\djmat\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

### **Baseline**

In [13]:
Qwen_baseline_test_answers = answers(test, generator=qwen)
print("✅ Qwen Baseline Test Answers Generation Completed!")
Qwen_baseline_blind_answers = answers(blind, generator=qwen)
print("✅ Qwen Baseline Blind Answers Generation Completed!")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔍 Starting generation for 2000 queries in Test set with Baseline setting...
✅ Generation completed for 2000 queries in Test set with Baseline setting.
✅ Qwen Baseline Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with Baseline setting...
✅ Generation completed for 1322 queries in Blind set with Baseline setting.
✅ Qwen Baseline Blind Answers Generation Completed!


### **RAG**

In [ ]:
Qwen_RAG_test_answers = answers(test, generator=qwen, settings='RAG', k=3)
print("✅ Qwen RAG Test Answers Generation Completed!")
Qwen_RAG_blind_answers = answers(blind, generator=qwen, settings='RAG', k=3)
print("✅ Qwen RAG Blind Answers Generation Completed!")

### **Oracle**

In [ ]:
Qwen_Oracle_test_answers = answers(test, generator=qwen, settings='Oracle', k=3)
print("✅ Qwen Oracle Test Answers Generation Completed!")
Qwen_Oracle_blind_answers = answers(blind, generator=qwen, settings='Oracle', k=3)
print("✅ Qwen Oracle Blind Answers Generation Completed!")

## **RAG output Files**

In [15]:
def save_rag_jsonl(answers_dict, filename):
    full_path = os.path.join(OUTPUTS_DIR, filename)
    with open(full_path, 'w', encoding='utf-8') as f:
        for q_id in answers_dict:
            json.dump(answers_dict[q_id], f, ensure_ascii=False)
            f.write('\n')
    print(f"💾 RAG file saved: {full_path}")

save_rag_jsonl(Llama_RAG_test_answers, "hit_k_1_-all-test-Llama-3.2-3B-Instruct.jsonl")
save_rag_jsonl(Llama_RAG_blind_answers, "hit_k_1_-all-blind-Llama-3.2-3B-Instruct.jsonl")
# save_rag_jsonl(Qwen_RAG_test_answers, "hit_k_1_-all-test-Qwen3-0.6B.jsonl")
# save_rag_jsonl(Qwen_RAG_blind_answers, "hit_k_1_-all-blind-Qwen3-0.6B.jsonl")

💾 RAG file saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1_-all-test-Llama-3.2-3B-Instruct.jsonl
💾 RAG file saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1_-all-blind-Llama-3.2-3B-Instruct.jsonl
